In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="xPHsHiUNZMnqk24t67JV")
project = rf.workspace("makineproje").project("no-entry-ysuzn")
version = project.version(1)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 96.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to no-entry-1 in yolov8:: 100%|██████████| 1219/1219 [00:00<00:00, 9642.15it/s]


In [ ]:
!pip install ultralytics
import ultralytics
ultralytics.checks()
from ultralytics import YOLO

Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.3/112.6 GB disk)


In [ ]:
import os
import random
import shutil
import yaml

# Rutas base
dataset_path = '/content/no-entry-1'
train_img_path = os.path.join(dataset_path, 'train/images')
train_lbl_path = os.path.join(dataset_path, 'train/labels')
val_img_path = os.path.join(dataset_path, 'valid/images')
val_lbl_path = os.path.join(dataset_path, 'valid/labels')
yaml_path = os.path.join(dataset_path, 'data.yaml')

# Crear directorios de validación si no existen
os.makedirs(val_img_path, exist_ok=True)
os.makedirs(val_lbl_path, exist_ok=True)

# Obtener lista de imágenes en train
images = [f for f in os.listdir(train_img_path) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Seleccionar 100 imágenes al azar para validación
num_to_move = min(100, len(images))
val_samples = random.sample(images, num_to_move)

print(f"Moviendo {num_to_move} archivos de train a valid...")

for img_name in val_samples:
    # Mover imagen
    shutil.move(os.path.join(train_img_path, img_name), os.path.join(val_img_path, img_name))

    # Mover etiqueta correspondiente (mismo nombre y extensión .txt)
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    src_lbl = os.path.join(train_lbl_path, lbl_name)
    if os.path.exists(src_lbl):
        shutil.move(src_lbl, os.path.join(val_lbl_path, lbl_name))

# Actualizar el archivo data.yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Asegurar que las rutas apunten correctamente
data['train'] = os.path.join(dataset_path, 'train/images')
data['val'] = os.path.join(dataset_path, 'valid/images')

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("Dataset listo y data.yaml actualizado.")

Moviendo 100 archivos de train a valid...
Dataset listo y data.yaml actualizado.


In [ ]:
!yolo task=detect mode=train model=yolov8n.pt data=/content/no-entry-1/data.yaml epochs=40 imgsz=320 plots=True

Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/no-entry-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

Validación

In [ ]:
!yolo val model="/content/runs/detect/train/weights/best.pt" data=/content/no-entry-1/data.yaml

Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 178.3±255.3 MB/s, size: 4.4 KB)
val: Scanning /content/no-entry-1/valid/labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 20.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 5.6it/s 1.2s
                   all        100        100      0.988          1      0.995      0.986
Speed: 0.8ms preprocess, 2.6ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to /content/runs/detect/val
💡 Learn more at https://docs.ultralytics.com/modes/val


Predicción

In [ ]:
!yolo predict model="/content/runs/detect/train/weights/best.pt" source='/content/aula2.png' imgsz=320

Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

image 1/1 /content/aula2.png: 256x320 (no detections), 41.8ms
Speed: 1.8ms preprocess, 41.8ms inference, 0.6ms postprocess per image at shape (1, 3, 256, 320)
Results saved to /content/runs/detect/predict-4
💡 Learn more at https://docs.ultralytics.com/modes/predict
